# Olist Brazilian E-Commerce — Business Intelligence Analysis

## Project Overview
This notebook starts v0.1.0: **Data Cleaning & Data Integration**. The Olist dataset is relational, so we first inspect and validate each table before deciding how to join them.

**Execution environment:** GitHub Codespaces / VS Code. This notebook does not use Google Colab or Google Drive.


## 1. Setup Environment
Import the libraries needed for data manipulation and initial visualization.


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'Pandas version: {pd.__version__}')


Pandas version: 3.0.6


## 2. Project Paths
Resolve the repository root so the notebook works whether the Codespace opens the notebook from the repository root or from the `notebooks/` directory.


In [2]:
def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd, *cwd.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'notebooks').exists():
            return path
    raise FileNotFoundError(
        'Project root not found. Open this notebook inside the olist-bi-project repository.'
    )

ROOT = find_project_root()
RAW_DATA = ROOT / 'data' / 'raw'
PROCESSED_DATA = ROOT / 'data' / 'processed'

print(f'Project root : {ROOT}')
print(f'Raw data    : {RAW_DATA}')
print(f'Processed   : {PROCESSED_DATA}')


Project root : /workspaces/olist-bi-project
Raw data    : /workspaces/olist-bi-project/data/raw
Processed   : /workspaces/olist-bi-project/data/processed


## 3. Load Datasets
The Olist dataset contains 9 CSV tables. The raw files must already be placed in `data/raw/` before running this section.


In [ ]:
file_names = {
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

dfs = {}
missing_files = []

for name, filename in file_names.items():
    filepath = RAW_DATA / filename
    try:
        dfs[name] = pd.read_csv(filepath)
        print(f'Loaded {name:20s} | Shape: {dfs[name].shape}')
    except FileNotFoundError:
        missing_files.append(filename)
        print(f'File not found: {filename}')

print(f'\nTotal datasets loaded: {len(dfs)} / {len(file_names)}')
if missing_files:
    print(f'Missing files: {missing_files}')


Loaded orders               | Shape: (99441, 8)
Loaded order_items          | Shape: (112650, 7)
Loaded order_payments       | Shape: (103886, 5)
Loaded order_reviews        | Shape: (99224, 7)
Loaded customers            | Shape: (99441, 5)
Loaded sellers              | Shape: (3095, 4)
Loaded products             | Shape: (32951, 9)
Loaded geolocation          | Shape: (1000163, 5)
Loaded category_translation | Shape: (71, 2)

Total datasets loaded: 9 / 9


## 4. Quick Overview per Table
Before cleaning anything, inspect row counts, columns, data types, sample records, and missing values. The findings here will determine the cleaning strategy for each table.


In [4]:
for name, df in dfs.items():
    print('\n' + '=' * 70)
    print(f'DATASET: {name.upper()}')
    print('=' * 70)
    print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
    print(f'Columns: {list(df.columns)}')
    print('\nData types:')
    print(df.dtypes.value_counts().to_string())
    print('\nFirst 2 rows:')
    print(df.head(2))
    print('\nMissing values (%):')
    missing = (df.isna().mean() * 100).sort_values(ascending=False)
    print(missing[missing > 0].to_string() if (missing > 0).any() else 'No missing values')



DATASET: ORDERS
Shape: 99,441 rows x 8 columns
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Data types:
str    8

First 2 rows:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08-07 15:27:45   

  order_estimated_delivery_date  
0           2017-10-18 00:00:00  
1           2018-08-13 00:00:00  

Missin

The initial profile shows that the 9 tables have different structures and data-quality characteristics. Several timestamp fields are currently loaded as strings, while missing values are concentrated in `orders`, `order_reviews`, and `products`.

In `orders`, missing delivery and approval timestamps should be interpreted together with `order_status` rather than filled automatically. In `order_reviews`, missing comment text does not necessarily invalidate the review because review scores and metadata may still be available. In `products`, missing attributes should be assessed according to how each field will be used in later analysis.

These findings indicate that cleaning should be evidence-based and table-specific rather than applying a single missing-value rule across the entire dataset.


## 5. Summary of All Tables


In [5]:
table_info = pd.DataFrame({
    'Table': list(dfs.keys()),
    'Rows': [df.shape[0] for df in dfs.values()],
    'Columns': [df.shape[1] for df in dfs.values()]
})

table_info


,Table,Rows,Columns
0,orders,99441,8
1,order_items,112650,7
2,order_payments,103886,5
3,order_reviews,99224,7
4,customers,99441,5
5,sellers,3095,4
6,products,32951,9
7,geolocation,1000163,5
8,category_translation,71,2


The 9 Olist tables have different row counts and therefore should not be assumed to share the same grain. `orders` contains 99,441 records, while `order_items` and `order_payments` contain more rows than `orders`. This indicates that these tables should not be treated as one-row-per-order tables without further validation.

The `geolocation` table is substantially larger than the transactional tables, with more than one million records. It should be treated as a separate location-level reference table until its grain and relationship to transactional data are validated.

These differences are important when designing the integration strategy because inappropriate joins could multiply rows and distort aggregated metrics.


## 6. Primary Key Uniqueness Check
Validate the identifiers that are expected to be unique before joining tables. A non-unique key can multiply rows during a merge.


In [6]:
pk_checks = {
    'orders': 'order_id',
    'customers': 'customer_id',
    'sellers': 'seller_id',
    'products': 'product_id'
}

for table, pk in pk_checks.items():
    if table in dfs:
        df = dfs[table]
        n_rows = len(df)
        n_unique = df[pk].nunique(dropna=True)
        n_missing = df[pk].isna().sum()
        duplicates = n_rows - n_unique - n_missing
        status = 'VALID' if duplicates == 0 and n_missing == 0 else 'REVIEW'

        print(
            f'{table:15s} | {pk:20s} | '
            f'{n_rows:7,} rows | {n_unique:7,} unique | '
            f'{n_missing:5,} missing | {duplicates:5,} duplicates | '
            f'{status}'
        )


orders          | order_id             |  99,441 rows |  99,441 unique |     0 missing |     0 duplicates | VALID
customers       | customer_id          |  99,441 rows |  99,441 unique |     0 missing |     0 duplicates | VALID
sellers         | seller_id            |   3,095 rows |   3,095 unique |     0 missing |     0 duplicates | VALID
products        | product_id           |  32,951 rows |  32,951 unique |     0 missing |     0 duplicates | VALID


The expected entity identifiers in `orders`, `customers`, `sellers`, and `products` are unique, with no missing or duplicate key values detected in the initial assessment.

This supports using these identifiers as entity-level keys when validating relationships with other tables. However, primary-key uniqueness does not prove that the entire table is free of duplicate rows; full-row duplicate validation remains a separate data-quality check.


## 7. Cross-Table Relationship Validation
Start with a basic relationship check: every `order_id` in `order_items` should have a corresponding record in `orders`. Additional relationships will be validated after we understand the actual table grains.


In [7]:
if {'orders', 'order_items'}.issubset(dfs):
    orders_ids = set(dfs['orders']['order_id'])
    items_order_ids = set(dfs['order_items']['order_id'])

    matched_order_ids = items_order_ids & orders_ids
    orphan_order_ids = items_order_ids - orders_ids

    total_item_order_ids = len(items_order_ids)
    matched_count = len(matched_order_ids)
    orphan_count = len(orphan_order_ids)
    coverage = matched_count / total_item_order_ids * 100 if total_item_order_ids else 0

    print(f'Orders unique IDs           : {len(orders_ids):,}')
    print(f'Order item unique order IDs : {total_item_order_ids:,}')
    print(f'Matched order IDs           : {matched_count:,}')
    print(f'Orphan order IDs            : {orphan_count:,}')
    print(f'Relationship coverage       : {coverage:.2f}%')
else:
    print('Load orders and order_items before running this validation.')


Orders unique IDs           : 99,441
Order item unique order IDs : 98,666
Matched order IDs           : 98,666
Orphan order IDs            : 0
Relationship coverage       : 100.00%


All distinct `order_id` values found in `order_items` are present in the `orders` table, resulting in 100% relationship coverage and zero orphan order IDs.

The number of unique order IDs in `order_items` is lower than the number of records in `orders`. This difference does not by itself indicate a data-quality problem because the validation is performed from the `order_items` side: every order referenced by an item must exist in the parent `orders` table.

This confirms that `order_id` is a valid relationship key for the validated `orders` → `order_items` relationship, while the two tables retain different grains.


## 8. Relationship & Grain Validation

Validate composite keys, cross-table relationships, and table cardinality before performing analytical joins. These checks distinguish expected one-to-many structures from actual key or relationship problems.

The validation is designed to preserve each table's natural grain and prevent row multiplication during later integration.


In [ ]:
def validation_result(check, status, detail):
    print(f'{status:7s} | {check}')
    print(f'         {detail}')


# 1. order_items composite key
order_item_key = ['order_id', 'order_item_id']
order_item_duplicates = dfs['order_items'].duplicated(order_item_key).sum()

validation_result(
    'order_items composite key',
    'PASS' if order_item_duplicates == 0 else 'REVIEW',
    f'{order_item_duplicates:,} duplicate rows for (order_id, order_item_id)'
)


# 2. order_payments composite key
payment_key = ['order_id', 'payment_sequential']
payment_duplicates = dfs['order_payments'].duplicated(payment_key).sum()

validation_result(
    'order_payments composite key',
    'PASS' if payment_duplicates == 0 else 'REVIEW',
    f'{payment_duplicates:,} duplicate rows for (order_id, payment_sequential)'
)


# 3. Review ID uniqueness
review_duplicates = dfs['order_reviews']['review_id'].duplicated().sum()

validation_result(
    'order_reviews review_id uniqueness',
    'PASS' if review_duplicates == 0 else 'REVIEW',
    f'{review_duplicates:,} duplicate review_id values'
)


# 4. Review cardinality per order
reviews_per_order = dfs['order_reviews'].groupby('order_id').size()
multiple_review_orders = (reviews_per_order > 1).sum()
max_reviews_per_order = reviews_per_order.max()

validation_result(
    'order_reviews cardinality',
    'INFO',
    f'{multiple_review_orders:,} orders have multiple review records; '
    f'maximum reviews per order = {max_reviews_per_order}'
)


# 5. customer_unique_id -> customer_id cardinality
customer_mapping = (
    dfs['customers']
    .groupby('customer_unique_id')['customer_id']
    .nunique()
)
multiple_customer_ids = (customer_mapping > 1).sum()

validation_result(
    'customer_unique_id cardinality',
    'INFO',
    f'{multiple_customer_ids:,} customer_unique_id values map to multiple customer_id values'
)


# 6. orders.customer_id -> customers.customer_id
order_customer_ids = set(dfs['orders']['customer_id'].dropna())
customer_ids = set(dfs['customers']['customer_id'].dropna())
orphan_customers = order_customer_ids - customer_ids

validation_result(
    'orders -> customers relationship',
    'PASS' if len(orphan_customers) == 0 else 'REVIEW',
    f'{len(orphan_customers):,} orphan customer_id values'
)


# 7. order_items.product_id -> products.product_id
item_product_ids = set(dfs['order_items']['product_id'].dropna())
product_ids = set(dfs['products']['product_id'].dropna())
orphan_products = item_product_ids - product_ids

validation_result(
    'order_items -> products relationship',
    'PASS' if len(orphan_products) == 0 else 'REVIEW',
    f'{len(orphan_products):,} orphan product_id values'
)


# 8. order_items.seller_id -> sellers.seller_id
item_seller_ids = set(dfs['order_items']['seller_id'].dropna())
seller_ids = set(dfs['sellers']['seller_id'].dropna())
orphan_sellers = item_seller_ids - seller_ids

validation_result(
    'order_items -> sellers relationship',
    'PASS' if len(orphan_sellers) == 0 else 'REVIEW',
    f'{len(orphan_sellers):,} orphan seller_id values'
)


# 9. Product category translation coverage
product_categories = set(
    dfs['products']['product_category_name'].dropna().unique()
)
translated_categories = set(
    dfs['category_translation']['product_category_name'].dropna().unique()
)
untranslated_categories = product_categories - translated_categories
coverage = (
    len(product_categories - untranslated_categories) / len(product_categories) * 100
    if product_categories else 0
)

validation_result(
    'product category translation coverage',
    'INFO',
    f'{coverage:.2f}% coverage; {len(untranslated_categories):,} untranslated categories'
)


# 10. Geolocation ZIP cardinality
geo_zip = dfs['geolocation'].groupby('geolocation_zip_code_prefix').size()
duplicate_geo_zip = (geo_zip > 1).sum()
max_geo_records = geo_zip.max()

validation_result(
    'geolocation ZIP cardinality',
    'INFO',
    f'{duplicate_geo_zip:,} ZIP prefixes have multiple geolocation records; '
    f'maximum records per ZIP prefix = {max_geo_records}'
)


### Validation Interpretation

A `PASS` result indicates that the tested key or relationship has no detected structural violation. An `INFO` result describes an expected or analytically important cardinality characteristic that should be documented rather than treated as an error. A `REVIEW` result indicates that the relationship or key requires investigation before it is used for analytical joins.

Composite keys are validated separately from entity-level primary keys because `order_items` and `order_payments` contain multiple records per order. Review and customer identity cardinality are also measured explicitly because these tables do not necessarily represent a strict one-to-one relationship with orders or customer entities.

The geolocation table is intentionally treated as a reference table with non-unique ZIP prefixes. Repeated ZIP prefixes are therefore reported as cardinality information rather than automatically classified as duplicates.

No analytical join should be performed until the relevant relationship has passed validation or its cardinality has been explicitly documented.


### 8.1 Review ID Duplicate Investigation

The `review_id` field is not unique in the source data. The investigation therefore checks whether repeated identifiers represent exact duplicate rows or distinct review records, and validates `(review_id, order_id)` as the review-level composite key.


In [ ]:
reviews = dfs['order_reviews'].copy()

# Identify review_id values that occur more than once.
review_id_counts = reviews['review_id'].value_counts()
duplicate_review_ids = review_id_counts[review_id_counts > 1]

duplicate_reviews = reviews[
    reviews['review_id'].isin(duplicate_review_ids.index)
].copy()

# Check whether repeated review_id values contain identical or conflicting records.
non_id_columns = [col for col in reviews.columns if col != 'review_id']

group_distinct_records = (
    duplicate_reviews
    .groupby('review_id')[non_id_columns]
    .apply(lambda group: len(group.drop_duplicates()))
)

exact_duplicate_groups = (group_distinct_records == 1).sum()
conflicting_duplicate_groups = (group_distinct_records > 1).sum()

# Check whether a repeated review_id is associated with multiple orders.
orders_per_review = duplicate_reviews.groupby('review_id')['order_id'].nunique()
multi_order_review_ids = (orders_per_review > 1).sum()

# Validate the proposed composite key: (review_id, order_id).
review_order_key = ['review_id', 'order_id']
review_order_duplicates = reviews.duplicated(review_order_key).sum()

print(f'Duplicate review_id values          : {len(duplicate_review_ids):,}')
print(f'Rows involved in duplicate IDs      : {len(duplicate_reviews):,}')
print(f'Exact duplicate review groups       : {exact_duplicate_groups:,}')
print(f'Conflicting review groups            : {conflicting_duplicate_groups:,}')
print(f'IDs linked to multiple order_id      : {multi_order_review_ids:,}')
print(f'Duplicate (review_id, order_id) keys : {review_order_duplicates:,}')

print('\nProposed review key validation:')
if review_order_duplicates == 0:
    print('PASS    | (review_id, order_id) is unique across order_reviews')
else:
    print('REVIEW  | (review_id, order_id) still contains duplicate combinations')

print('\nSample duplicate review_id values:')
sample_ids = duplicate_review_ids.head(10).index
print(
    duplicate_reviews[
        duplicate_reviews['review_id'].isin(sample_ids)
    ].sort_values(['review_id', 'order_id']).to_string(index=False)
)


### Validation Interpretation

The validation confirms that the expected composite keys for `order_items` and `order_payments` are unique, and that the tested foreign-key relationships contain no orphan identifiers. These results support their use in subsequent analytical joins while preserving the original table grains.

The `order_reviews` table requires special treatment. `review_id` alone is not unique: 789 identifiers are repeated across distinct review records and are associated with multiple orders. The investigation found no exact duplicate review groups. The combination `(review_id, order_id)` is unique across the table, so it is the validated composite key for review-level analysis.

Multiple review records per order, multiple `customer_id` values associated with the same `customer_unique_id`, repeated geolocation records per ZIP prefix, and incomplete category translation are documented as cardinality or coverage characteristics rather than automatically classified as data-quality errors.

No records are removed based on these validation results. Analytical joins should only be performed after the grain and cardinality of the participating tables have been explicitly documented.

**Section 8 conclusion:** the validated keys and relationships are structurally suitable for the next cleaning and integration stage, with the documented grain exceptions preserved as part of the data model.


## 9. Initial Findings & Cleaning Strategy

The initial assessment shows that the Olist dataset contains multiple tables with different grains, data types, and data-quality characteristics. The cleaning process will therefore be based on evidence observed in each table rather than applying the same transformation to all datasets.

### Initial Findings

- All 9 source tables were loaded successfully.
- The expected entity identifiers in `orders`, `customers`, `sellers`, and `products` are unique, with no missing or duplicate key values detected.
- The validated `orders` → `order_items` relationship has 100% coverage from `order_items` to `orders`, with no orphan `order_id` values detected.
- Several timestamp columns are currently stored as strings and require conversion to datetime.
- Missing values are concentrated in `orders`, `order_reviews`, and `products`, but they should not be removed or imputed automatically.
- `order_items` and `order_payments` contain more rows than `orders`, confirming that table grain must be considered during integration.
- The `geolocation` table contains more than one million records and requires separate grain validation before being used in analytical joins.

### Cleaning Strategy

Based on these findings, the next stage will focus on:

1. Converting relevant date and timestamp columns to appropriate datetime types.
2. Investigating missing values in `orders` according to order status and delivery lifecycle.
3. Preserving review records with missing comment text while retaining available review scores and metadata.
4. Investigating missing product attributes before deciding whether to retain, impute, or exclude specific fields.
5. Checking duplicate rows separately from primary-key uniqueness.
6. Validating additional cross-table relationships before performing analytical joins.
7. Preparing cleaned tables in `data/processed/` while preserving the original raw data.

### Integration Principle

No large multi-table merge will be performed until the grain, key, and relationship of the participating tables have been validated.


## 10. Data Cleaning

The validation stage is complete. Cleaning now begins from the validated structure, while the original `dfs` data remains unchanged. Cleaned tables are stored separately in `cleaned_dfs` so that every transformation can be traced and reviewed before the processed data is exported.

The first cleaning stage focuses on two evidence-based transformations: converting date and timestamp fields to appropriate datetime types, and auditing exact duplicate rows. Missing values will be handled separately according to the business meaning of each field rather than using a blanket rule.


### 10.1 Datetime Standardization

Convert fields that represent dates or timestamps to pandas datetime types. Invalid values are converted to `NaT` and reported so that the conversion does not silently hide data-quality problems.


In [ ]:
cleaned_dfs = {name: df.copy() for name, df in dfs.items()}

datetime_columns = {
    'orders': [
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ],
    'order_items': ['shipping_limit_date'],
    'order_reviews': [
        'review_creation_date',
        'review_answer_timestamp',
    ],
}

for table, columns in datetime_columns.items():
    for column in columns:
        before_missing = cleaned_dfs[table][column].isna().sum()
        cleaned_dfs[table][column] = pd.to_datetime(
            cleaned_dfs[table][column],
            errors='coerce'
        )
        after_missing = cleaned_dfs[table][column].isna().sum()
        newly_invalid = after_missing - before_missing

        print(
            f'{table:15s} | {column:35s} | '
            f'dtype={str(cleaned_dfs[table][column].dtype):20s} | '
            f'newly invalid={newly_invalid:,}'
        )


The datetime conversion preserves missing timestamps as missing values. Any newly invalid values are explicitly reported instead of being silently imputed. The original raw-loaded tables remain available in `dfs` for comparison.


### 10.2 Exact Duplicate Row Audit

Check complete-row duplicates independently from primary-key validation. A duplicate primary key does not necessarily mean an exact duplicate row, as demonstrated by the `order_reviews` investigation.


In [ ]:
duplicate_summary = []

for name, df in cleaned_dfs.items():
    duplicate_rows = df.duplicated().sum()
    duplicate_summary.append({
        'table': name,
        'rows': len(df),
        'exact_duplicate_rows': duplicate_rows,
    })

duplicate_summary = pd.DataFrame(duplicate_summary)
duplicate_summary


Exact duplicate rows are reported before any deletion is performed. The audit found no exact duplicate rows in the core transactional and dimension tables. The `geolocation` table contains 261,831 exact duplicate rows, but geolocation remains outside the current core integration scope. These rows are therefore **not removed at this stage** and will be investigated only if location-based analysis requires the table.

This keeps the current cleaning scope focused on tables that directly support the core order, customer, seller, product, payment, and review analysis while preserving the raw geolocation structure for future use.

### Cleaning Principle

Raw data is preserved in `dfs`, while transformations are applied to `cleaned_dfs`. Each cleaning decision must be supported by the earlier schema, grain, and relationship validation. No duplicate rows are removed solely because they are technically identical without considering the table's analytical grain and current project scope.


## 11. Missing Value Assessment

Datetime conversion and exact duplicate auditing are complete. The next cleaning stage assesses missing values in the core analytical tables before deciding whether each field should be retained as missing, imputed, excluded from a specific analysis, or otherwise transformed.

Missing values will be interpreted according to field meaning and business context. No blanket imputation or row deletion will be applied.


### 11.1 Missing Value Profile

Create a consistent missing-value profile for the cleaned tables. The profile reports both the number and percentage of missing values so that high-missingness fields can be investigated without automatically treating them as errors.


In [ ]:
core_tables = [
    'orders',
    'order_items',
    'order_payments',
    'order_reviews',
    'customers',
    'sellers',
    'products',
    'category_translation',
]

missing_profile = []

for table in core_tables:
    df = cleaned_dfs[table]
    for column in df.columns:
        missing_count = df[column].isna().sum()
        if missing_count > 0:
            missing_profile.append({
                'table': table,
                'column': column,
                'missing_count': missing_count,
                'missing_pct': missing_count / len(df) * 100,
            })

missing_profile = (
    pd.DataFrame(missing_profile)
    .sort_values(['table', 'missing_pct'], ascending=[True, False])
    .reset_index(drop=True)
)

missing_profile


### 11.2 Missing Value Decision Framework

The missing-value profile will be reviewed by table and field meaning. In `orders`, missing delivery timestamps are expected to depend on order lifecycle and status. In `order_reviews`, missing comment text does not invalidate a review score. In `products`, missing descriptive or physical attributes require field-specific treatment based on their later analytical use.

The next step is to investigate these missing fields and define an explicit treatment for each one. Until that decision is documented, missing values remain unchanged in `cleaned_dfs`.


### 11.3 Missing Value Investigation

Before applying any treatment, investigate the business context and missing-value patterns for the affected fields. The objective is to distinguish structurally expected missing values from fields that may require a later cleaning rule.


In [ ]:
# 1. Order delivery fields by order status
order_missing_by_status = (
    cleaned_dfs['orders']
    .groupby('order_status')[[
        'order_delivered_customer_date',
        'order_delivered_carrier_date',
        'order_approved_at',
    ]]
    .apply(lambda df: df.isna().sum())
)

order_missing_by_status


In [ ]:
# 2. Product missing-value overlap
product_missing_columns = [
    'product_category_name',
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
]

product_missing_overlap = (
    cleaned_dfs['products'][product_missing_columns]
    .isna()
    .sum(axis=1)
    .value_counts()
    .sort_index()
)

product_missing_overlap


In [ ]:
# 3. Review comment missingness versus review score
review_missing_by_score = (
    cleaned_dfs['order_reviews']
    .groupby('review_score')[[
        'review_comment_title',
        'review_comment_message',
    ]]
    .apply(lambda df: df.isna().sum())
)

review_missing_by_score


These investigations are descriptive only. No missing values are filled or removed yet. Treatment decisions will be made after reviewing the resulting patterns and considering the analytical role of each field.


### 11.4 Missing Value Treatment Decisions

The investigation shows that most missing values are structurally related to the lifecycle or optional nature of the fields rather than evidence that rows should be discarded. Treatment is therefore defined at field level, with missing values preserved when they carry business meaning.


In [ ]:
# Inspect products with missing attributes across the selected product fields.
product_mask = cleaned_dfs['products'][product_missing_columns].isna()
missing_count_per_product = product_mask.sum(axis=1)

product_missing_pattern = (
    cleaned_dfs['products'].loc[
        missing_count_per_product > 0,
        ['product_id']
    ].copy()
)

product_missing_pattern['missing_count'] = missing_count_per_product.loc[
    missing_count_per_product > 0
].to_numpy()

product_missing_pattern['missing_fields'] = product_mask.loc[
    missing_count_per_product > 0
].apply(lambda row: ', '.join(row.index[row].tolist()), axis=1).to_numpy()

product_missing_pattern.sort_values(
    ['missing_count', 'product_id'],
    ascending=[False, True]
)


**Treatment decisions**

| Table / Field | Decision | Rationale |
| --- | --- | --- |
| `orders` delivery timestamps | Preserve missing values | Missingness follows order lifecycle/status and should not be fabricated with imputation. |
| `orders.order_approved_at` | Preserve missing values | Missing records are concentrated in non-standard order states; no blanket imputation is justified. |
| `order_reviews` comment title/message | Preserve missing values | Comments are optional text fields; `review_score` remains usable when comments are absent. |
| `products` descriptive attributes | Preserve missing values | The missing-value pattern is concentrated in a small subset of products; missing product attributes do not justify row deletion. |
| `products` physical attributes | Preserve missing values | Very low missingness does not justify arbitrary imputation without a domain rule. |

The product overlap output should be read as **the number of missing fields per product**, not the number of missing values in a specific column. The observed pattern therefore needs to be interpreted from the `missing_count` and `missing_fields` columns before any transformation is considered.

No rows are deleted and no missing values are imputed at this stage. The cleaned tables therefore retain the original missingness while using standardized datetime types.


The missing-value assessment is complete for the current core tables. The next cleaning stage will focus on data-type and value consistency checks before preparing the validated cleaned tables for export.


## 12. Data Type & Value Consistency Checks

The missing-value assessment is complete. This stage checks whether values are internally consistent with the expected domain, range, and lifecycle rules of each core table. These checks are diagnostic only: no values are changed and no rows are removed until the results have been reviewed.


### 12.1 Orders Status & Timestamp Consistency

Validate order-status values and identify timestamp sequences that are logically inconsistent. Missing timestamps are not treated as errors here because their treatment was already documented in Section 11.


In [ ]:
orders = cleaned_dfs['orders']

valid_order_statuses = sorted(orders['order_status'].dropna().unique().tolist())
expected_order_statuses = [
    'approved', 'canceled', 'created', 'delivered',
    'invoiced', 'processing', 'shipped', 'unavailable'
]

unexpected_statuses = sorted(
    set(valid_order_statuses) - set(expected_order_statuses)
)

print(f'Unique order_status values : {valid_order_statuses}')
print(f'Unexpected status values   : {unexpected_statuses}')

timestamp_rules = {
    'approved_before_carrier': (
        'order_approved_at', 'order_delivered_carrier_date'
    ),
    'purchase_before_approved': (
        'order_purchase_timestamp', 'order_approved_at'
    ),
    'carrier_before_customer': (
        'order_delivered_carrier_date', 'order_delivered_customer_date'
    ),
    'purchase_before_customer': (
        'order_purchase_timestamp', 'order_delivered_customer_date'
    ),
    'estimated_after_purchase': (
        'order_purchase_timestamp', 'order_estimated_delivery_date'
    ),
}

timestamp_violations = []
for rule, (start_col, end_col) in timestamp_rules.items():
    mask = (
        orders[start_col].notna()
        & orders[end_col].notna()
        & (orders[start_col] > orders[end_col])
    )
    timestamp_violations.append({
        'rule': rule,
        'violations': int(mask.sum()),
    })

pd.DataFrame(timestamp_violations)


### 12.2 Transactional Numeric Consistency

Check non-negative monetary values and basic item/payment quantity rules. Zero values are reported separately from negative values because a zero can be valid in some analytical contexts.


In [ ]:
items = cleaned_dfs['order_items']
payments = cleaned_dfs['order_payments']

numeric_checks = pd.DataFrame([
    {
        'table': 'order_items',
        'field': 'price',
        'negative': int((items['price'] < 0).sum()),
        'zero': int((items['price'] == 0).sum()),
    },
    {
        'table': 'order_items',
        'field': 'freight_value',
        'negative': int((items['freight_value'] < 0).sum()),
        'zero': int((items['freight_value'] == 0).sum()),
    },
    {
        'table': 'order_payments',
        'field': 'payment_value',
        'negative': int((payments['payment_value'] < 0).sum()),
        'zero': int((payments['payment_value'] == 0).sum()),
    },
    {
        'table': 'order_payments',
        'field': 'payment_installments',
        'negative': int((payments['payment_installments'] < 0).sum()),
        'zero': int((payments['payment_installments'] == 0).sum()),
    },
])

numeric_checks


In [ ]:
item_id_checks = pd.DataFrame({
    'metric': [
        'non_positive_order_item_id',
        'max_order_item_id',
    ],
    'value': [
        int((items['order_item_id'] <= 0).sum()),
        int(items['order_item_id'].max()),
    ],
})

item_id_checks


### 12.3 Payment & Review Domain Checks

Validate categorical and ordinal fields against their expected domain values before any downstream aggregation.


In [ ]:
valid_payment_types = [
    'credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined'
]

unexpected_payment_types = sorted(
    set(payments['payment_type'].dropna().unique())
    - set(valid_payment_types)
)

reviews = cleaned_dfs['order_reviews']

review_checks = pd.DataFrame([
    {
        'check': 'unexpected_payment_type',
        'count': len(unexpected_payment_types),
        'detail': str(unexpected_payment_types),
    },
    {
        'check': 'non_positive_payment_installments',
        'count': int((payments['payment_installments'] <= 0).sum()),
        'detail': 'Expected installments > 0',
    },
    {
        'check': 'review_score_outside_1_to_5',
        'count': int((~reviews['review_score'].between(1, 5)).sum()),
        'detail': 'Expected review_score between 1 and 5',
    },
])

review_checks


### 12.4 ZIP Prefix & Product Attribute Checks

Validate ZIP-prefix ranges and identify negative or zero product attributes. These checks are descriptive; zero values are not automatically classified as invalid without considering the source field's meaning.


In [ ]:
customer_zip = cleaned_dfs['customers']['customer_zip_code_prefix']
seller_zip = cleaned_dfs['sellers']['seller_zip_code_prefix']

zip_checks = pd.DataFrame([
    {
        'table': 'customers',
        'field': 'customer_zip_code_prefix',
        'outside_00000_99999': int((~customer_zip.between(0, 99999)).sum()),
    },
    {
        'table': 'sellers',
        'field': 'seller_zip_code_prefix',
        'outside_00000_99999': int((~seller_zip.between(0, 99999)).sum()),
    },
])

zip_checks


In [ ]:
products = cleaned_dfs['products']
product_numeric_fields = [
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
]

product_checks = pd.DataFrame([
    {
        'field': field,
        'negative': int((products[field] < 0).sum()),
        'zero': int((products[field] == 0).sum()),
    }
    for field in product_numeric_fields
])

product_checks


In [ ]:
translation = cleaned_dfs['category_translation']
translation_key_duplicates = translation.duplicated(
    subset=['product_category_name']
).sum()

print(f'Duplicate category translation keys: {translation_key_duplicates:,}')


### 12.5 Validation Interpretation

Section 12 is diagnostic only. The results must be reviewed before any value correction, recoding, or row removal is introduced. A zero count indicates no detected violation for the tested rule; non-zero counts require contextual interpretation rather than automatic cleaning.

No transformations are applied in this section.


### 12.6 Non-Zero Result Investigation

The consistency checks identified several non-zero counts. These cases are investigated before any correction is considered. The objective is to distinguish genuine data-quality anomalies from valid characteristics of the source data.


**A. Order timestamp violations**

Inspect affected orders by status and compare the relevant timestamps. Missing timestamps are excluded from the violation masks.


In [ ]:
orders = cleaned_dfs['orders'].copy()

approved_carrier_mask = (
    orders['order_approved_at'].notna()
    & orders['order_delivered_carrier_date'].notna()
    & (orders['order_approved_at'] > orders['order_delivered_carrier_date'])
)

carrier_customer_mask = (
    orders['order_delivered_carrier_date'].notna()
    & orders['order_delivered_customer_date'].notna()
    & (orders['order_delivered_carrier_date'] > orders['order_delivered_customer_date'])
)

timestamp_investigation = pd.DataFrame({
    'rule': ['approved_after_carrier', 'carrier_after_customer_delivery'],
    'violations': [int(approved_carrier_mask.sum()), int(carrier_customer_mask.sum())],
})

status_breakdown = pd.concat([
    orders.loc[approved_carrier_mask, 'order_status'].value_counts().rename('approved_after_carrier'),
    orders.loc[carrier_customer_mask, 'order_status'].value_counts().rename('carrier_after_customer_delivery'),
], axis=1).fillna(0).astype(int)

timestamp_investigation
status_breakdown


In [ ]:
timestamp_samples = orders.loc[
    approved_carrier_mask | carrier_customer_mask,
    ['order_id', 'order_status', 'order_purchase_timestamp',
     'order_approved_at', 'order_delivered_carrier_date',
     'order_delivered_customer_date', 'order_estimated_delivery_date']
].copy()

timestamp_samples.head(20)


**B. Zero payment installments and zero payment value**

Inspect the complete payment records with zero installments or zero payment value. These values are rare and should not be altered without confirming their payment context.


In [ ]:
zero_installments = payments.loc[
    payments['payment_installments'] == 0,
    ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
].copy()

zero_payment_value = payments.loc[
    payments['payment_value'] == 0,
    ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
].copy()

zero_installments
zero_payment_value


In [ ]:
payment_zero_installment_context = payments.loc[
    payments['payment_installments'] == 0,
    [
      'order_id',
      'payment_sequential',
      'payment_type',
      'payment_installments',
      'payment_value'
    ]
  ].copy()

payment_zero_installment_context


**C. Zero freight value and zero product weight**

Zero freight and zero product weight are reviewed separately because they can represent valid source values rather than negative or impossible measurements.


In [ ]:
zero_freight = items.loc[
    items['freight_value'] == 0,
    ['order_id', 'order_item_id', 'product_id', 'seller_id', 'price', 'freight_value']
].copy()

zero_weight = products.loc[
    products['product_weight_g'] == 0,
    ['product_id', 'product_category_name', 'product_weight_g',
     'product_length_cm', 'product_height_cm', 'product_width_cm']
].copy()

print(f'Zero freight records: {len(zero_freight):,}')
print(f'Zero product-weight records: {len(zero_weight):,}')

zero_freight.head(20)
zero_weight


### 12.7 Validation Interpretation

The non-zero consistency results were reviewed against the business meaning and observed record context. The findings are classified as preserved source characteristics or data-quality anomalies rather than automatically corrected values.

| Finding | Count | Decision |
| --- | ---: | --- |
| Approved timestamp occurs after carrier timestamp | 1,359 | Preserve source values and flag as temporal inconsistency |
| Carrier timestamp occurs after customer delivery timestamp | 23 | Preserve source values and flag as temporal inconsistency |
| Zero payment installments | 2 | Preserve source values and flag as payment anomaly |
| Zero payment value | 9 | Preserve; observed on `voucher` and `not_defined` payment records |
| Zero freight value | 383 | Preserve as a valid zero-valued source field unless analysis requires otherwise |
| Zero product weight | 4 | Preserve and flag as suspicious product attribute |
| Negative monetary/product values | 0 | No correction required |
| Invalid review scores | 0 | No correction required |
| ZIP prefixes outside tested range | 0 | No correction required |
| Duplicate category translation keys | 0 | No correction required |

Timestamp inconsistencies and suspicious zero-valued attributes are retained because the source does not provide sufficient evidence to reconstruct the correct values. Downstream analytical datasets may use explicit quality flags or exclude affected records only when the analytical definition requires it.

Section 12 is therefore complete. No source values are overwritten and no rows are deleted as a result of these checks.


## 13. Final Cleaning Rules

The validation and investigation stages are complete. This section applies only transformations directly supported by the evidence gathered above. The original `dfs` tables remain unchanged, while `cleaned_dfs_final` becomes the processed analytical layer.


### 13.1 Cleaning Rules

The final cleaning rules are intentionally conservative:

1. Preserve the original raw tables in `dfs`.
2. Use `cleaned_dfs_final` as the final transformation layer.
3. Keep standardized pandas datetime types from Section 10.
4. Preserve missing values documented in Section 11; no blanket imputation is applied.
5. Preserve exact duplicate rows because none were detected in core tables; geolocation duplicates remain outside the core cleaning scope.
6. Preserve temporal inconsistencies, zero payment values, zero freight values, and suspicious product weights rather than inventing replacement values.
7. Preserve the two zero-installment payment records and document them as quality anomalies.
8. Do not remove rows solely because a value is missing or technically unusual.
9. Prepare cleaned tables for export without overwriting source data.


In [ ]:
cleaned_dfs_final = {name: df.copy() for name, df in cleaned_dfs.items()}

quality_flags = {
    'orders_timestamp_inconsistency': orders.loc[
        approved_carrier_mask | carrier_customer_mask,
        ['order_id']
    ].copy(),
    'payment_installment_anomaly': payments.loc[
        payments['payment_installments'] == 0,
        ['order_id', 'payment_sequential']
    ].copy(),
    'product_zero_weight': products.loc[
        products['product_weight_g'] == 0,
        ['product_id']
    ].copy(),
}

{
    'tables': len(cleaned_dfs_final),
    'quality_flag_groups': len(quality_flags),
    'rows_removed': 0,
    'missing_values_imputed': 0,
}


### 13.2 Final Cleaning Validation

Before export, verify that row counts remain unchanged and that the expected datetime fields remain datetime types. This confirms that the final cleaning stage did not silently remove records or alter source grain.


In [ ]:
final_validation = []

for name, original in dfs.items():
    cleaned = cleaned_dfs_final[name]
    final_validation.append({
        'table': name,
        'original_rows': len(original),
        'cleaned_rows': len(cleaned),
        'row_count_changed': len(original) != len(cleaned),
    })

final_validation = pd.DataFrame(final_validation)
final_validation


### 13.3 Cleaning Conclusion

The final cleaning layer applies conservative, traceable transformations only. Missing values and source anomalies that cannot be reliably reconstructed are preserved and documented. The resulting tables are ready for controlled export to `data/processed/` while the raw source remains preserved in `data/raw/`.
